In [1]:
pip install librosa numpy

Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
import librosa

def hz_to_note_name(hz):
    """周波数を音名（MIDIノート番号）に変換する関数"""
    if hz <= 0: return None
    # MIDIノート番号を算出
    midi_note = 69 + 12 * np.log2(hz / 440.0)
    return int(round(midi_note))

def analyze_melody(file_path):
    # 1. 音声の読み込み
    y, sr = librosa.load(file_path)

    # 2. ピッチトラッキング (短時間フーリエ変換ベース)
    # fmin, fmaxはカエルの歌が出る音域に合わせて調整してください
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr, fmin=200, fmax=800)

    # 各フレームの主要なピッチを抽出
    detected_notes = []
    for t in range(pitches.shape[1]):
        index = magnitudes[:, t].argmax()
        pitch = pitches[index, t]
        if pitch > 0:
            note = hz_to_note_name(pitch)
            # 連続する同じ音はスキップして音の変化を捉える
            if not detected_notes or detected_notes[-1] != note:
                detected_notes.append(note)
    return detected_notes

# 3. 判定ロジック
# カエルの歌(ハ長調): ドレミファ ミファミレ ドレミファ ミファミレ...
# MIDIノート番号: ド=60, レ=62, ミ=64, ファ=65, ソ=67
target_melody = [62]

# ファイルパスを指定して解析
file_path = "レ.wav"  # 用意した録音ファイル
played_melody = analyze_melody(file_path)

print(f"検出された音階(MIDI): {played_melody}")

# 単純な一致判定
matches = sum(1 for p, t in zip(played_melody, target_melody) if p == t)
accuracy = (matches / len(target_melody)) * 100
print(f"正解率: {accuracy:.2f}%")

検出された音階(MIDI): [62]
正解率: 100.00%
